<img src="https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/images/edrai_logo.png" alt="EDR|AI" width="300"/>

# Chapter 42 — Survey Experiments: Vignettes, Conjoint, and List Experiments

This is the **companion notebook** of [Chapter 42 — Survey Experiments: Vignettes, Conjoint, and List Experiments](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/part7-further-routes/survey-experiments.html) from **EDR|AI — Evidence-Driven Research in the Age of AI**. Authored by [Davi Moreira](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/index.html).

[Open the chapter](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/part7-further-routes/survey-experiments.html) · [Book home](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/index.html) · [Verification Guide](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/verification-guide.html)

*AI is your arm and your research assistant, not your brain.*

## How to use this notebook

1. Work top to bottom, with the chapter open in another tab.
2. Copy each **AI prompt** into your AI tool, run it, then record in the response cell what came back and what you verified.
3. Run the code cells; change something and run again.
4. Finish the **It is your turn** workspace at the end — that is this chapter's step of your own research project.
5. Log every AI use in your **AI Research Ledger**: task · tool · prompt · output summary · decision · verification method · remaining concern · you as the responsible researcher.
6. Your AI can be more than a chatbot: agentic tools can run multi-step work for you. Delegating boldly is fine; reviewing, curating, and deciding stay yours.

> **The research decision.** Decide what you randomize inside your questionnaire,
> and whether the number it yields describes something people already hold (a
> prevalence, a preference) or tests the effect of a message you wrote. Then say
> what a stated answer can and cannot tell you about what people actually do. The
> questionnaire is yours, so no tool gets to decide what its numbers mean.

## Code from the chapter

The cells below come from the chapter. Run them, then change something and run again — the numbers should move the way the chapter says they will.

*From the section “A worked example”.* **What this cell does:** exactly what the chapter walks through in that section; run it and compare with the chapter.

In [ ]:
import numpy as np
SEED = 464
rng = np.random.default_rng(SEED)

n, truth = 100_000, 0.20   # TRUE share who pasted client data into an unapproved AI tool
trait = rng.random(n) < truth
longer = rng.permutation(n) < n // 2      # coin flip: who sees the five-item list

def run_list(control):                    # control: one yes/no column per item
    base = control.sum(axis=1)
    top = base.max()                      # highest count the control items allow
    # top + 1 on the longer list would be a confession, so anyone there says top
    count = base + (longer & trait) - (longer & trait & (base == top))
    est = count[longer].mean() - count[~longer].mean()
    se = np.sqrt(count[longer].var(ddof=1) / longer.sum()
                 + count[~longer].var(ddof=1) / (~longer).sum())
    return est, se, top, (base == top).mean()

popular = rng.random((n, 4)) < np.array([0.85, 0.80, 0.90, 0.75])
drives = rng.random(n) < 0.5              # "I drove to work" vs "I took transit"
balanced = np.column_stack([drives, ~drives, rng.random(n) < 0.15,
                            rng.random(n) < 0.70])

print(f"true prevalence                 : {truth*100:.1f}%")
for name, items in (("popular control items", popular),
                    ("balanced control items", balanced)):
    est, se, top, ceiling = run_list(items)
    print(f"{name:<23} : {est*100:5.1f}% +/- {1.96*se*100:.1f}"
          f"   (at the top count of {top}: {ceiling*100:.0f}%)")
print("\nthe list did not fail at random. it failed where the design let a")
print("truthful answer become a confession")

**Reading the output.** The chapter reads this output in the same section; check yours against it, then change one input and rerun. The numbers should move the way the chapter says they will.

*From the section “A seeded simulation”.* **What this cell does:** exactly what the chapter walks through in that section; run it and compare with the chapter.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 464
rng = np.random.default_rng(SEED)

# LEFT: a list experiment vs a direct question. True prevalence is 20%,
# but only 45% of those with the trait admit it when asked directly.
truth, admit = 0.20, 0.45
control_p = np.array([0.5, 0.3, 0.6, 0.2])     # four innocuous control items
sizes = [300, 600, 1200, 2400, 4800]
rows = []
for n in sizes:
    trait = rng.random(n) < truth
    listed = rng.permutation(n) < n // 2        # half get the longer list
    count = (rng.random((n, 4)) < control_p).sum(axis=1) + (listed & trait)
    lst = count[listed].mean() - count[~listed].mean()
    lst_se = np.sqrt(count[listed].var(ddof=1) / listed.sum()
                     + count[~listed].var(ddof=1) / (~listed).sum())
    said_yes = trait & (rng.random(n) < admit)  # the same n, asked directly
    d = said_yes.mean()
    rows.append((n, lst, lst_se, d, np.sqrt(d * (1 - d) / n)))

# RIGHT: a conjoint of job offers. Each respondent picks one of two offers,
# five times; every attribute level is dealt at random.
R, T = 800, 5
salary = rng.integers(0, 3, (R, T, 2))          # +0%, +10%, +20%
remote = rng.integers(0, 3, (R, T, 2))          # 0, 2, 5 remote days a week
ai_ok = rng.integers(0, 2, (R, T, 2))           # AI tools banned / allowed
util = (np.array([0, .35, .90])[salary] + np.array([0, .30, .55])[remote]
        + .15 * ai_ok + rng.logistic(0, 1, (R, T, 2)))
chosen = (util == util.max(axis=2, keepdims=True)).astype(float)
levels = [("Salary +10%", salary, 1), ("Salary +20%", salary, 2),
          ("Remote 2 days", remote, 1), ("Remote 5 days", remote, 2),
          ("AI tools allowed", ai_ok, 1)]

def amces(ix):
    return [chosen[ix][a[ix] == k].mean() - chosen[ix][a[ix] == 0].mean()
            for _, a, k in levels]

est = np.array(amces(np.arange(R)))
boot = np.array([amces(rng.integers(0, R, R)) for _ in range(300)])  # resample PEOPLE
lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)

for n, l, lse, d, dse in rows:
    print(f"n = {n:>5}  list {l*100:5.1f}% +/- {1.96*lse*100:4.1f}   "
          f"direct {d*100:5.1f}% +/- {1.96*dse*100:4.1f}")
for (name, _, _), e, a, b in zip(levels, est, lo, hi):
    print(f"{name:<17}: {e*100:+5.1f} pp  [{a*100:+5.1f}, {b*100:+5.1f}]")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.6, 3.4))
x = np.arange(len(sizes))
ax1.errorbar(x - .1, [r[1] * 100 for r in rows], [1.96 * r[2] * 100 for r in rows],
             fmt="o", color="#2a78d6", label="list experiment")
ax1.errorbar(x + .1, [r[3] * 100 for r in rows], [1.96 * r[4] * 100 for r in rows],
             fmt="s", color="#eb6834", label="direct question")
ax1.axhline(truth * 100, color="#333333", ls="--", lw=1)
ax1.set_ylim(-20, 50)
ax1.text(.03, .04, "true prevalence = 20%", fontsize=8, transform=ax1.transAxes)
ax1.set_xticks(x, [str(n) for n in sizes])
ax1.set_xlabel("Respondents")
ax1.set_ylabel("Estimated prevalence (%)")
ax1.legend(fontsize=8, frameon=False, loc="upper right")
y = np.arange(len(levels))[::-1]
ax2.errorbar(est * 100, y, xerr=[(est - lo) * 100, (hi - est) * 100],
             fmt="o", color="#2a78d6")
ax2.axvline(0, color="#333333", lw=.8)
ax2.set_yticks(y, [name for name, _, _ in levels])
ax2.set_xlabel("AMCE (percentage points)")
fig.tight_layout()
plt.show()

**Reading the output.** The chapter reads this output in the same section; check yours against it, then change one input and rerun. The numbers should move the way the chapter says they will.

## It is your turn

<!-- station-pointer:begin -->
> **A further route beyond the five pathways.** This lesson
> extends [Studio 5: Develop the pathway](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/studios/studio05-develop-the-pathway.html). Read it once
> you have declared your primary pathway and your question
> calls for this design. Studio 5's milestone asks for the
> same decisions, answered for this route.
<!-- station-pointer:end -->

*Your design is declared and diagnosed. This route is for a question that a
questionnaire can answer only if chance decides what each respondent sees, and it
asks you to say out loud what kind of number that produces.*

The hands-on half of this section lives in the chapter's **companion notebook**: open it in Colab with the badge at the top, and work the steps there.

Commit your own answer first, then delegate. Each prompt is a checkable job, not a
request for a verdict. Work them as a loop. The first answer is a draft: find the
claim you cannot check, say so in your next message, and run it again. Some tools
will draft a whole questionnaire, field it to simulated respondents, and hand you a
finished analysis. The finished look is exactly what makes the questions in this
chapter worth asking before you accept it.

> **Do not delegate.**
>
> Three calls stay yours. You decide **what you randomize and which kind of number it
> yields**, a prevalence, a preference, or the effect of a message. You decide **the
> wording and the menu**: the sensitive item, the control items, the attribute
> levels, because every estimate inherits them. And you decide **whether a stated
> answer can stand in for behaviour in your write-up, and whether you may ask at
> all**. A tool can draft items and attack your design. What the numbers mean is
> your call.

**Step 1.** Name what you randomize and the kind of answer it yields. Write your Research
Contract's question line, then one line that says whether the design is a list,
conjoint, vignette, or framing experiment, and whether the inquiry is
descriptive (prevalence or preference) or causal (the effect of a message, or of
a profile feature on stated choices), with its reach. Let the words of your
question decide the kind, not the fact that you randomized.

✍️ **Your work for step 1.** Double-click this cell and write your answer here.

**Step 2.** Write your estimand and your warrant in two sentences. The estimand names the
quantity, the respondents, and the setting: "the share of this firm's staff who
did X last month" or "the change in the chance an offer is chosen when it
carries level L, under the attribute mix in the attached table." The warrant
names what licenses each crossing: random assignment of versions for the
comparison, and your sampling for the reach.

*When you are ready to delegate this step:*

```text
Act as a survey methodologist. Here is my design: [paste question, instrument
type, sample source, and estimand]. List, as a table, every assumption my
estimate rests on, for example no design effects and no liars for a list
experiment, or the attribute distribution for a conjoint. For each, say what I
could check in my own data or pilot.
```

After running, verify: compare the table against the assumptions you wrote first
and against this chapter; if the tool missed the sampling assumption behind your
reach, add it back. Counters **illusion of completeness** (a tidy table that
omits the one assumption your reach depends on).

✍️ **Your work for step 2.** Double-click this cell and write your answer here.

✍️ **Your run.** Double-click this cell and record: what the AI returned (one or two lines), what you verified and how, and your ledger row.

**Step 3.** Draft the instrument yourself before any tool sees it. For a list experiment,
write the sensitive item and four control items. For a conjoint, write the
attribute table with every level. For a vignette or framing design, write both
versions word for word.

**Red-team your instrument.**

```text
Here are my control items and sensitive item: [paste]. Act as a hostile
reviewer. Estimate roughly what share of respondents could reach the highest or
lowest count my control items allow (remember that two items that cannot both be
true lower the highest count), name any item that could change how people count the
others, and say which item a respondent might find threatening. Do not rewrite
the list for me; list the problems so I fix them myself.
```

After running, verify: if it says the list looks fine, push back with "assume
ceiling effects are severe; which item causes them?" Praise without an objection
is a red flag. Counters **sycophantic agreement** (praise that reviews your ego,
not your evidence).

✍️ **Your work for step 3.** Double-click this cell and write your answer here.

✍️ **Your run.** Double-click this cell and record: what the AI returned (one or two lines), what you verified and how, and your ledger row.

**Step 4.** Diagnose the design by simulation before you field it. In the companion
notebook, set a plausible true value, your planned sample size, and your actual
items or levels, and record the interval width your design will produce.

*When you are ready to delegate this step:*

```text
Act as a Python tutor. Here is the simulation from my companion notebook with my
own items and sample size: [paste the cell]. Explain line by line how the
interval for the difference in mean counts is computed, and tell me what would
change if I asked the direct question to everyone instead.
```

After running, verify: rerun the cell yourself and confirm every number the
tool quotes appears in your own printout. Counters **confident fabrication**
(a fluent walkthrough quoting an interval your code never produced).

✍️ **Your work for step 4.** Double-click this cell and write your answer here.

✍️ **Your run.** Double-click this cell and record: what the AI returned (one or two lines), what you verified and how, and your ledger row.

**Step 5.** Write your limits line. Say whether the estimate describes stated or revealed
behaviour, what demand effect is most plausible for your respondents, what you
will do with failed attention or manipulation checks, and your permission status
from the ethics chapter.

✍️ **Your work for step 5.** Double-click this cell and write your answer here.

**Step 6.** Log the step in your AI Research Ledger, and verify at least one output with a
named method from the [Verification Guide](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/verification-guide.html). A
**simulation** is the natural check for this route: plant a known prevalence or
a known AMCE, run your exact design, and confirm it recovers the planted value
within its interval. **Direct calculation** is the strong second check:
recompute one difference in means by hand from your pilot counts. An AI reviewer
may run the checks with you; the decision to accept or reject stays yours.

✍️ **Your work for step 6.** Double-click this cell and write your answer here.

### The standard this section is held to

Use this as a self-check while you work. It is also the bar the same work meets later, once your project carries it. Each row: **0** missing, **1** attempted but incomplete, generic, or unverified, **2** complete, specific to your own project, and verified where a check applies. **14 points in all.**

| # | Criterion | 0–2 |
|---|---|---|
| Step 1 | Name what you randomize and the kind of answer it yields | |
| Step 2 | Write your estimand and your warrant in two sentences | |
| Step 3 | Draft the instrument yourself before any tool sees it | |
| Step 4 | Diagnose the design by simulation before you field it | |
| Step 5 | Write your limits line | |
| Step 6 | Log the step in your AI Research Ledger, and verify at least one output with a named method from the Verification Guide | |
| + | Craft and verification record: AI use logged in your AI Research Ledger, claims stated with their uncertainty, and each key claim verified with a named method | |

In [ ]:
# Scratch space — use this cell for any code your steps need.

**Before you leave this notebook:** add today's rows to your AI Research Ledger, and verify your key claim with a named method from the [Verification Guide](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/verification-guide.html). AI can review AI — but the last decision is human.

Next: [Chapter 43 — Audit Studies: Testing How Institutions and AI Systems Treat Cases](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/part7-further-routes/audit-studies.html). That chapter may not be on your route — [Studio 5: Develop the pathway](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/studios/studio05-develop-the-pathway.html) is the junction; follow the lesson that matches your own project.